# CodeGuide: Discovering Thermally Stable Li-Ion Battery Separator Polymers

This notebook walks through the entire project codebase so you can understand how each piece works.

**Problem:** Every lithium-ion battery contains a thin polymer separator that prevents the electrodes from short-circuiting. If this film melts or shrinks during overheating, the battery can catch fire. Most commercial separators use PE or PP, which melt at just 130-165°C — a known weak point during thermal runaway.

**Approach:** Train a KNN classifier on 39 known separator polymers to predict safety from material properties, then screen 101 untested polymers to discover new candidates.

### Project Pipeline

| Step | Script | What It Does |
|------|--------|--------------|
| 1a | `scrape_celgard.py` | Scrape Celgard product page for separator product info + PDF links |
| 1a | `scrape_celgard_pdfs.py` | Download & parse Celgard PDF datasheets for separator specs |
| 1b | `scrape_literature.py` | Scrape PMC review articles for separator comparison tables |
| 1c | `explore_polymers.py` | Load & explore the OpenPoly database (741 polymers, 26 properties) |
| 2 | `build_training_data.py` | Compile 39 known separator polymers with safety labels + material properties |
| 3 | `train_model.py` | Train KNN classifier, tune hyperparameters, screen 101 new polymers |
| 4 | `clustering_and_evaluation.py` | K-Means clustering + cross-validation confusion matrix |
| 5 | `plot_candidates.py` | Visualize top candidates + radar chart comparison |

---
## Part 1a: Scraping Celgard Product Data
`scrape_celgard.py`

Celgard is the largest manufacturer of battery separators. We scrape their product page to identify what separator products exist and get links to their PDF datasheets.

**Key concepts:**
- `requests.get()` fetches the HTML of a web page
- `BeautifulSoup` parses that HTML so we can search it with `.find_all()`
- We extract product names, descriptions, PDF links, and categories
- Categories are determined by walking through the HTML tree and tracking which `<h3>` heading each product falls under

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Scrape the Celgard product data page
url = "https://www.celgard.com/product-data"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

print(f"Status code: {response.status_code}")

# Find all product entries on the page
products = soup.find_all("div", attrs={"class": "ds-item"})
print(f"Found {len(products)} products")

# Let's look at the structure of one product entry
print("\nFirst product entry:")
print(products[0])

In [ ]:
# Extract product info: name, description, and PDF link
product_data = []
for product in products:
    info = {}

    # Get the product name
    title = product.find("div", attrs={"class": "ds-title"})
    info["name"] = title.string.strip() if title else None

    # Get the description
    desc = product.find("div", attrs={"class": "ds-desc"})
    if desc:
        p_tag = desc.find("p")
        info["description"] = p_tag.string.strip() if p_tag and p_tag.string else None

    # Get the PDF download link
    link = product.find("a", attrs={"class": "ds-download"})
    if link:
        info["pdf_url"] = "https://www.celgard.com" + link.attrs["href"]

    product_data.append(info)

df_celgard = pd.DataFrame(product_data)

# Extract the category each product belongs to
# All products are inside one <div class="datasheets"> container.
# Categories are <h3> tags between groups of <div class="ds-item"> entries.
# We walk through the children and track which <h3> heading we're under.
container = soup.find("div", attrs={"class": "datasheets"})

product_categories = []
current_category = None
for child in container.children:
    if child.name == "h3":
        current_category = child.get_text().strip()
    elif child.name == "div" and "ds-item" in child.get("class", []):
        title = child.find("div", attrs={"class": "ds-title"})
        if title:
            product_categories.append({
                "name": title.string.strip(),
                "category": current_category
            })

df_categories = pd.DataFrame(product_categories)
df_celgard = df_celgard.merge(df_categories, on="name", how="left")

print(f"Scraped {len(df_celgard)} Celgard products:")
df_celgard

In [ ]:
# Save to CSV
df_celgard.to_csv("data/celgard_products.csv", index=False)
print("Saved to data/celgard_products.csv")

### Part 1a (continued): Scraping Celgard PDF Datasheets
`scrape_celgard_pdfs.py`

Now we download each PDF and extract the separator specs (porosity, thickness, Gurley number, tensile strength, puncture strength).

**Key concepts:**
- `pdfplumber` opens PDFs and extracts tables/text
- Each Celgard PDF has a table with 3 columns: `[Property, Unit, Value]`
- We add `time.sleep(0.2)` between requests to be polite to the server

In [ ]:
import pdfplumber
import time
import os
import io

# Load the product list we already scraped
df_celgard = pd.read_csv("data/celgard_products.csv")
print(f"Found {len(df_celgard)} products to download PDFs for\n")

# Create a directory to store downloaded PDFs
os.makedirs("data/celgard_pdfs", exist_ok=True)

# Download each PDF and extract text/tables
all_specs = []

for _, row in df_celgard.iterrows():
    name = row["name"]
    pdf_url = row["pdf_url"]
    category = row["category"]

    print(f"Downloading: {name}...")

    # Download the PDF
    response = requests.get(pdf_url)
    if response.status_code != 200:
        print(f"  FAILED (status {response.status_code})")
        continue

    # Save locally
    filename = pdf_url.split("/")[-1]
    filepath = f"data/celgard_pdfs/{filename}"
    with open(filepath, "wb") as f:
        f.write(response.content)

    # Parse the PDF with pdfplumber
    pdf = pdfplumber.open(io.BytesIO(response.content))

    # Extract tables from all pages
    tables_found = []
    for page in pdf.pages:
        tables = page.extract_tables()
        for table in tables:
            tables_found.extend(table)

    # Also extract raw text for fallback
    text = ""
    for page in pdf.pages:
        text += page.extract_text() + "\n"

    # Parse the table rows into a dict
    # Each PDF table has 3 columns: [Property, Unit, Value]
    specs = {"name": name, "category": category, "pdf_file": filename}
    for table_row in tables_found:
        if table_row and len(table_row) >= 3:
            key = str(table_row[0]).strip() if table_row[0] else ""
            unit = str(table_row[1]).strip() if table_row[1] else ""
            val = str(table_row[2]).strip() if table_row[2] else ""
            # Skip header row and empty rows
            if key and val and key != "Basic Film Properties":
                specs[key] = val
                specs[key + " (unit)"] = unit

    all_specs.append(specs)
    pdf.close()

    # Be polite - stagger requests
    time.sleep(0.2)

# Convert to DataFrame
df_specs = pd.DataFrame(all_specs)

print(f"\n=== Extracted specs for {len(df_specs)} products ===")
print(f"Columns found: {list(df_specs.columns)}")
df_specs

In [ ]:
# Save
df_specs.to_csv("data/celgard_specs.csv", index=False)
print("Saved to data/celgard_specs.csv")

---
## Part 1b: Scraping Literature from PMC Review Articles
`scrape_literature.py`

We scrape open-access PMC review articles for their comparison tables containing separator polymer properties (porosity, thermal stability, tensile strength, ionic conductivity, etc.).

**Key concepts:**
- PMC (PubMed Central) hosts open-access scientific papers with HTML tables we can parse
- We target specific tables from 7 review articles that compare separator polymers
- Tables are extracted by finding all `<table>` tags and parsing `<tr>` rows
- Tables with 5+ rows are saved as CSVs for later use

In [ ]:
# --- Articles to scrape ---
articles = [
    {"id": "PMC10534950", "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC10534950/",
     "title": "Engineering Polymer Separator Membranes",
     "target_table": 7},
    {"id": "PMC11241740", "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC11241740/",
     "title": "Cellulose-Based Separators",
     "target_table": 1},
    {"id": "PMC11511470", "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC11511470/",
     "title": "Electrospun PVDF Separators",
     "target_table": 1},
    {"id": "PMC6161240", "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC6161240/",
     "title": "PVDF and Copolymers for Separators",
     "target_table": 1},
    {"id": "PMC12073824", "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC12073824/",
     "title": "PPS Separators",
     "target_table": 0},
    {"id": "PMC7831081", "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC7831081/",
     "title": "Li-Ion Battery Separator Safety",
     "target_table": 0},
    {"id": "PMC7603034", "url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC7603034/",
     "title": "Functional Separators for Li Metal Battery",
     "target_table": 0},
]

all_tables = {}

for article in articles:
    print(f"\n{'='*60}")
    print(f"Scraping: {article['title']}")
    print(f"URL: {article['url']}")

    response = requests.get(article["url"])
    print(f"Status: {response.status_code}")

    if response.status_code != 200:
        print("  SKIPPED - could not access")
        continue

    soup = BeautifulSoup(response.text, "html.parser")

    # Find all tables on the page
    tables = soup.find_all("table")
    print(f"Found {len(tables)} tables")

    # Extract each table into a list of rows
    for t_idx, table in enumerate(tables):
        rows = table.find_all("tr")
        if len(rows) < 3:
            continue

        # Extract headers
        headers = []
        header_row = rows[0]
        for th in header_row.find_all(["th", "td"]):
            headers.append(th.get_text().strip())

        if len(headers) < 3:
            continue

        # Extract data rows
        data_rows = []
        for row in rows[1:]:
            cells = row.find_all(["td", "th"])
            row_data = [cell.get_text().strip() for cell in cells]
            if len(row_data) >= 2:
                data_rows.append(row_data)

        if len(data_rows) < 2:
            continue

        print(f"\n  Table {t_idx}: {len(data_rows)} rows, {len(headers)} cols")
        print(f"  Headers: {headers[:6]}...")

        # Store the table
        table_key = f"{article['id']}_table{t_idx}"
        all_tables[table_key] = {
            "article": article["title"],
            "article_id": article["id"],
            "table_index": t_idx,
            "headers": headers,
            "data": data_rows,
            "n_rows": len(data_rows)
        }

    # Be polite to PMC servers
    time.sleep(0.5)

# --- Summary ---
print(f"\n{'='*60}")
print(f"SUMMARY: Extracted {len(all_tables)} tables total")
print(f"{'='*60}")

total_rows = 0
for key, table in all_tables.items():
    print(f"  {key}: {table['n_rows']} rows - {table['article']}")
    total_rows += table["n_rows"]
print(f"\nTotal data rows across all tables: {total_rows}")

In [ ]:
# --- Save the richest tables as CSVs ---
for key, table in all_tables.items():
    # Save tables with 5+ rows
    if table["n_rows"] >= 5:
        # Pad rows to match header length
        max_cols = max(len(table["headers"]), max(len(r) for r in table["data"]))
        headers = table["headers"] + [f"col_{i}" for i in range(max_cols - len(table["headers"]))]

        padded_data = []
        for row in table["data"]:
            padded_data.append(row + [""] * (max_cols - len(row)))

        df = pd.DataFrame(padded_data, columns=headers[:max_cols])
        filename = f"data/literature_{key}.csv"
        df.to_csv(filename, index=False)
        print(f"Saved {filename} ({table['n_rows']} rows)")

---
## Part 1c: Exploring the OpenPoly Polymer Database
`explore_polymers.py`

OpenPoly is a dataset of 741 polymers with 26 experimentally measured properties. This serves as our screening pool — polymers that have never been tested as battery separators.

**Key concepts:**
- We check data availability to pick which properties to use as model features
- The 6 best-covered properties become our feature set: Tg, Tm, Td (thermal) + Tensile Strength, Young's Modulus, Elongation at Break (mechanical)
- We identify which known separator polymers appear in OpenPoly
- Scatter plots show where known separators sit in property space vs. all polymers

In [ ]:
import pandas as pd
import numpy as np

# Load the OpenPoly dataset
df_polymers = pd.read_csv("data/OpenPoly_polymers.csv")
print(f"Shape: {df_polymers.shape}")
df_polymers.head()

In [ ]:
# Properties relevant to battery separators
separator_relevant = [
    "Name", "Tg (K)", "Tm (K)", "Td (K)",
    "Tensile Strength (MPa)", "Young's Modulus (MPa)",
    "Elongation at Break (%)", "Thermal Conductivity (W/m K)",
    "Crystallization Temperature (K)", "Water Contact Angle (deg)",
    "Water Uptake (%)", "Hardness (MPa)", "Compressive Strength (MPa)"
]

df_relevant = df_polymers[separator_relevant]

# How much data do we have for each property?
print("Data availability per property:")
print(df_relevant.drop(columns="Name").notna().sum().sort_values(ascending=False))

In [ ]:
# Focus on the best-covered properties for modeling
# These are the features we can use for both training and screening
key_features = ["Tg (K)", "Tm (K)", "Td (K)",
                "Tensile Strength (MPa)", "Young's Modulus (MPa)",
                "Elongation at Break (%)"]

# How many polymers have ALL of these properties?
df_complete = df_relevant.dropna(subset=key_features)
print(f"Polymers with all key properties: {len(df_complete)}")
df_complete

In [ ]:
# Let's look at the known separator polymers in this dataset
known_separator_names = ["PE", "PP", "PVDF", "PAN", "HDPE", "LDPE",
                         "polypropylene", "polyethylene",
                         "poly(vinylidene fluoride)", "polyacrylonitrile",
                         "cellulose", "PLA", "PET", "nylon", "PI",
                         "polyimide", "PTFE", "PVA"]

# Search for these in the Name column (case-insensitive)
mask = df_polymers["Name"].str.lower().isin([n.lower() for n in known_separator_names])
df_known = df_polymers[mask]
print(f"Found {len(df_known)} known separator-relevant polymers in OpenPoly:")
df_known[["Name"] + key_features]

In [ ]:
import matplotlib.pyplot as plt

# Scatter plot: Tg vs Tm for all polymers
ax = df_polymers.plot.scatter(x="Tg (K)", y="Tm (K)", alpha=0.4,
                              color="gray", label="All polymers")

# Highlight known separator polymers
df_known.plot.scatter(x="Tg (K)", y="Tm (K)", alpha=0.9,
                      color="red", label="Known separator polymers", ax=ax)

ax.set_title("Thermal Properties of Polymers")
ax.legend()
plt.show()

In [ ]:
# Scatter plot: Tensile Strength vs Young's Modulus
ax = df_polymers.plot.scatter(x="Young's Modulus (MPa)",
                              y="Tensile Strength (MPa)",
                              alpha=0.4, color="gray",
                              label="All polymers")

df_known.plot.scatter(x="Young's Modulus (MPa)",
                      y="Tensile Strength (MPa)",
                      alpha=0.9, color="red",
                      label="Known separator polymers", ax=ax)

ax.set_title("Mechanical Properties of Polymers")
ax.legend()
plt.show()

---
## Part 2: Building the Training Dataset
`build_training_data.py`

We combine our domain knowledge (which polymers are known separators and whether they're safe) with OpenPoly's material property data.

**Step 1:** Compile 42 known separator polymers with binary safety labels based on published thermal stability data:
- **safe** = maintains structural integrity above 150°C
- **unsafe** = loses structural integrity at or below 150°C

The 150°C threshold is based on industry abuse test standards (UL 2054, IEC 62133).

Mechanical features included in model because polymers can deform under heat before melting, even if the safety class is thermal based -- both thermal and mechanical pathways matter.

**Step 2:** Join with OpenPoly to get 6 material properties for each polymer. Only polymers with complete data are kept → 39 training polymers (25 safe, 14 unsafe).

In [ ]:
# --- Load data sources ---
df_polymers = pd.read_csv("data/OpenPoly_polymers.csv")

# ============================================================
# STEP 1: Compile known separator polymers with safety labels
# ============================================================

# Binary safety labels based on thermal stability from published literature:
#   "safe"   = maintains structural integrity above 150C
#   "unsafe" = loses structural integrity at or below 150C

separator_polymers = pd.DataFrame([
    # --- Polyolefins ---
    {"polymer": "PP", "shutdown_temp_C": 170, "safety": "safe"},
    {"polymer": "PE", "shutdown_temp_C": 135, "safety": "unsafe"},
    {"polymer": "UHMWPE", "shutdown_temp_C": 135, "safety": "unsafe"},
    {"polymer": "HDPE", "shutdown_temp_C": 130, "safety": "unsafe"},
    {"polymer": "LDPE", "shutdown_temp_C": 110, "safety": "unsafe"},

    # --- Fluoropolymers ---
    {"polymer": "PVDF", "shutdown_temp_C": 160, "safety": "safe"},
    {"polymer": "PTFE", "shutdown_temp_C": 327, "safety": "safe"},

    # --- High-performance polymers ---
    {"polymer": "PAN", "shutdown_temp_C": 300, "safety": "safe"},
    {"polymer": "PI", "shutdown_temp_C": 350, "safety": "safe"},
    {"polymer": "PPS", "shutdown_temp_C": 300, "safety": "safe"},
    {"polymer": "PEI", "shutdown_temp_C": 280, "safety": "safe"},
    {"polymer": "PES", "shutdown_temp_C": 220, "safety": "safe"},
    {"polymer": "PSF", "shutdown_temp_C": 185, "safety": "safe"},

    # --- Engineering polymers ---
    {"polymer": "PET", "shutdown_temp_C": 250, "safety": "safe"},
    {"polymer": "PA6", "shutdown_temp_C": 220, "safety": "safe"},
    {"polymer": "PC", "shutdown_temp_C": 230, "safety": "safe"},
    {"polymer": "POM", "shutdown_temp_C": 175, "safety": "safe"},
    {"polymer": "nylon", "shutdown_temp_C": 220, "safety": "safe"},

    # --- Hydrophilic / gel polymers ---
    {"polymer": "PVA", "shutdown_temp_C": 230, "safety": "safe"},
    {"polymer": "PMMA", "shutdown_temp_C": 105, "safety": "unsafe"},
    {"polymer": "PAA", "shutdown_temp_C": 200, "safety": "safe"},
    {"polymer": "PVP", "shutdown_temp_C": 60, "safety": "unsafe"},
    {"polymer": "PEG", "shutdown_temp_C": 60, "safety": "unsafe"},
    {"polymer": "PEO", "shutdown_temp_C": 65, "safety": "unsafe"},

    # --- Biodegradable / low-stability ---
    {"polymer": "PLA", "shutdown_temp_C": 160, "safety": "safe"},
    {"polymer": "PBS", "shutdown_temp_C": 115, "safety": "unsafe"},
    {"polymer": "PCL", "shutdown_temp_C": 60, "safety": "unsafe"},

    # --- Bio-based ---
    {"polymer": "cellulose", "shutdown_temp_C": 260, "safety": "safe"},
    {"polymer": "chitosan", "shutdown_temp_C": 200, "safety": "safe"},

    # --- Others ---
    {"polymer": "PS", "shutdown_temp_C": 100, "safety": "unsafe"},
    {"polymer": "PVC", "shutdown_temp_C": 100, "safety": "unsafe"},
    {"polymer": "PMP", "shutdown_temp_C": 235, "safety": "safe"},

    # --- Duplicate names in OpenPoly ---
    {"polymer": "polypropylene", "shutdown_temp_C": 170, "safety": "safe"},
    {"polymer": "polyethylene", "shutdown_temp_C": 135, "safety": "unsafe"},
    {"polymer": "polyimide", "shutdown_temp_C": 350, "safety": "safe"},
    {"polymer": "polycarbonate", "shutdown_temp_C": 230, "safety": "safe"},
    {"polymer": "polystyrene", "shutdown_temp_C": 100, "safety": "unsafe"},
    {"polymer": "poly(vinylidene fluoride)", "shutdown_temp_C": 160, "safety": "safe"},
    {"polymer": "polyacrylonitrile", "shutdown_temp_C": 300, "safety": "safe"},
    {"polymer": "polyethersulfone", "shutdown_temp_C": 220, "safety": "safe"},
    {"polymer": "polysulfone", "shutdown_temp_C": 185, "safety": "safe"},
    {"polymer": "cellulose acetate", "shutdown_temp_C": 230, "safety": "safe"},
])

print(f"Step 1: Compiled {len(separator_polymers)} separator polymers with safety labels")
separator_polymers["safety"].value_counts()

In [ ]:
# ============================================================
# STEP 2: Join with OpenPoly for material properties
# ============================================================

key_features = ["Tg (K)", "Tm (K)", "Td (K)",
                "Tensile Strength (MPa)", "Young's Modulus (MPa)",
                "Elongation at Break (%)"]

df_polymer_props = df_polymers[["Name"] + key_features].copy()

# Deduplicate: keep row with most non-null features per polymer
df_polymer_props["n_features"] = df_polymer_props[key_features].notna().sum(axis=1)
df_polymer_props = df_polymer_props.sort_values("n_features", ascending=False)
df_polymer_props = df_polymer_props.drop_duplicates(subset="Name", keep="first")
df_polymer_props = df_polymer_props.drop(columns="n_features")

df_training = separator_polymers.merge(
    df_polymer_props,
    left_on="polymer",
    right_on="Name",
    how="left"
)

# Drop rows without material properties
df_training = df_training.dropna(subset=key_features)
df_training = df_training.drop(columns="Name")

print(f"Step 2: {len(df_training)} polymers after joining with OpenPoly")
print(f"\nSafety Label Distribution:")
print(df_training["safety"].value_counts())
print(f"\nFeatures ({len(key_features)}):")
for f in key_features:
    n_valid = df_training[f].notna().sum()
    print(f"  {f}: {n_valid}/{len(df_training)} non-null")

In [ ]:
# View the training data
df_training[["polymer", "shutdown_temp_C", "safety"] + key_features]

In [ ]:
# Save training data
df_training.to_csv("data/39known_polymers(training_data).csv", index=False)
print("Saved to data/39known_polymers(training_data).csv")

---
## Part 3: Training the KNN Model & Screening
`train_model.py`

We build a pipeline with `StandardScaler` + `KNeighborsClassifier`, evaluate with cross-validation, tune hyperparameters with `GridSearchCV`, and then screen 101 untested polymers.

**Key concepts:**
- **StandardScaler**: Normalizes features to mean=0, std=1 so no single feature dominates due to scale differences (e.g., Td in hundreds of Kelvin vs. Elongation in %)
- **KNN**: Classifies a new polymer by looking at its k nearest neighbors in feature space and taking a majority vote
- **Pipeline**: Chains scaler + classifier so scaling is applied consistently during CV and prediction
- **Cross-validation (5-fold)**: Splits data into 5 folds, trains on 4, tests on 1, rotates — gives honest accuracy estimate
- **GridSearchCV**: Tries all combinations of k (1-11) and distance metric (euclidean, manhattan) to find the best hyperparameters
- **F1 macro**: Averages F1 across both classes equally, important because we have more safe (25) than unsafe (14) polymers

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import confusion_matrix, f1_score

# --- Step 1: Load training data ---
df_training = pd.read_csv("data/39known_polymers(training_data).csv")

key_features = ["Tg (K)", "Tm (K)", "Td (K)",
                "Tensile Strength (MPa)", "Young's Modulus (MPa)",
                "Elongation at Break (%)"]

X_train = df_training[key_features]
y_train = df_training["safety"]

print(f"Training set: {len(X_train)} polymers, {len(key_features)} features")
print(f"Labels: {y_train.value_counts().to_dict()}")

### Step 2: Build pipeline and cross-validate

The pipeline ensures that scaling is done **inside** cross-validation (fit on training folds, transform test fold). This prevents data leakage.

In [ ]:
# Pipeline: scale features, then classify with KNN
pipeline = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5, metric="euclidean")
)

# Cross-validation: accuracy
scores_acc = cross_val_score(
    pipeline, X_train, y_train,
    scoring="accuracy", cv=5
)
print(f"Accuracy:  {scores_acc.mean():.3f} (+/- {scores_acc.std():.3f})")

# Cross-validation: F1 macro (better for imbalanced classes)
scores_f1 = cross_val_score(
    pipeline, X_train, y_train,
    scoring="f1_macro", cv=5
)
print(f"F1 (macro): {scores_f1.mean():.3f} (+/- {scores_f1.std():.3f})")

### Step 3: GridSearchCV to find best k and distance metric

We search over:
- `n_neighbors`: 1 to 11
- `metric`: euclidean (straight-line distance) vs. manhattan (city-block distance)

The naming convention `kneighborsclassifier__n_neighbors` comes from sklearn's pipeline: `stepname__param`.

In [ ]:
grid_cv = GridSearchCV(
    pipeline,
    param_grid={
        "kneighborsclassifier__n_neighbors": range(1, 12),
        "kneighborsclassifier__metric": ["euclidean", "manhattan"],
    },
    scoring="f1_macro",
    cv=5
)

grid_cv.fit(X_train, y_train)
print(f"Best params: {grid_cv.best_params_}")
print(f"Best F1 (macro): {grid_cv.best_score_:.3f}")

### Model Selection Plot

This shows how F1 score changes with different values of k for both distance metrics. The annotated point marks the best combination.

In [ ]:
results = pd.DataFrame(grid_cv.cv_results_)
fig, ax = plt.subplots()
for metric in ["euclidean", "manhattan"]:
    mask = results["param_kneighborsclassifier__metric"] == metric
    subset = results[mask].sort_values("param_kneighborsclassifier__n_neighbors")
    subset.plot.line(
        x="param_kneighborsclassifier__n_neighbors",
        y="mean_test_score",
        label=metric,
        ax=ax
    )

# Label the best F1 point
best_k = grid_cv.best_params_["kneighborsclassifier__n_neighbors"]
best_metric = grid_cv.best_params_["kneighborsclassifier__metric"]
best_f1 = grid_cv.best_score_
ax.annotate(f"Best: k={best_k}, {best_metric}\nF1 = {best_f1:.3f}",
            xy=(best_k, best_f1),
            xytext=(best_k - 2, best_f1 - 0.06),
            fontsize=10, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="black"),
            bbox=dict(boxstyle="round,pad=0.3", facecolor="yellow", alpha=0.8))
ax.scatter([best_k], [best_f1], color="black", zorder=5, s=60)

ax.set_xticks(range(1, 12))
ax.set_xlabel("k (number of neighbors)")
ax.set_ylabel("F1 Score (macro)")
ax.set_title("Model Selection: F1 vs k")
ax.legend()
plt.show()

### Step 4: Training confusion matrix

The confusion matrix shows how the best model performs on the training data:
- **True Positives**: correctly predicted safe/unsafe
- **False Positives**: predicted safe but actually unsafe (dangerous!)
- **False Negatives**: predicted unsafe but actually safe (overly conservative)

Note: training accuracy is optimistic — the CV confusion matrix (Part 4) is more honest.

In [ ]:
best_pipeline = grid_cv.best_estimator_
y_train_pred = best_pipeline.predict(X_train)

cm = confusion_matrix(y_train, y_train_pred, labels=["safe", "unsafe"])
print("Confusion Matrix (rows=actual, cols=predicted):")
print(f"Labels: [safe, unsafe]")
print(cm)

# Per-class metrics
print("\nPer-Class Metrics:")
for label in ["safe", "unsafe"]:
    y_true_binary = (y_train == label)
    y_pred_binary = (y_train_pred == label)
    tp = (y_true_binary & y_pred_binary).sum()
    fp = (~y_true_binary & y_pred_binary).sum()
    fn = (y_true_binary & ~y_pred_binary).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 / (1/precision + 1/recall) if (precision > 0 and recall > 0) else 0
    print(f"  {label:10s}  precision={precision:.3f}  recall={recall:.3f}  F1={f1:.3f}")

### Step 5: Screen OpenPoly polymers for new separator candidates

We take the 101 polymers from OpenPoly that are NOT in our training set (and have complete features), and predict whether each would be a safe or unsafe separator.

`predict_proba()` gives us a confidence score — e.g., if 5/5 nearest neighbors are "safe", the probability is 1.0.

In [ ]:
df_polymers = pd.read_csv("data/OpenPoly_polymers.csv")

# Get polymers with complete features that are NOT in our training set
base_features = ["Tg (K)", "Tm (K)", "Td (K)",
                 "Tensile Strength (MPa)", "Young's Modulus (MPa)",
                 "Elongation at Break (%)"]
training_names = df_training["polymer"].str.lower().tolist()
df_screen = df_polymers.dropna(subset=base_features).copy()
df_screen = df_screen[~df_screen["Name"].str.lower().isin(training_names)]

print(f"Polymers available to screen: {len(df_screen)}")

X_screen = df_screen[key_features]
df_screen["predicted_safety"] = best_pipeline.predict(X_screen)
df_screen["predicted_proba_safe"] = best_pipeline.predict_proba(X_screen)[:,
    list(best_pipeline.classes_).index("safe")]

# Sort by probability of being safe
df_screen = df_screen.sort_values("predicted_proba_safe", ascending=False)

print(f"\nScreening Summary:")
print(df_screen["predicted_safety"].value_counts())

In [ ]:
# Top 20 predicted safe candidates
top_safe = df_screen[df_screen["predicted_safety"] == "safe"].head(20)
top_safe[["Name", "predicted_safety", "predicted_proba_safe"] + key_features]

In [ ]:
# Save full screening results
df_screen.to_csv("data/101screening_polymers.csv", index=False)
print("Saved to data/101screening_polymers.csv")

### Screening Results Plot

This scatter plot shows where screened polymers sit in Tg vs. Tm space, alongside the known training polymers. Color encodes known/predicted and safe/unsafe.

In [ ]:
import plotly.express as px

# Build combined DataFrame for plotly
df_screen_plot = df_screen[["Tg (K)", "Tm (K)", "Name", "predicted_safety"]].copy()
df_screen_plot["group"] = "Predicted " + df_screen_plot["predicted_safety"]

df_train_plot = df_training[["Tg (K)", "Tm (K)", "polymer", "safety"]].copy()
df_train_plot = df_train_plot.rename(columns={"polymer": "Name"})
df_train_plot["group"] = "Known " + df_train_plot["safety"]

df_all_plot = pd.concat([df_screen_plot, df_train_plot], ignore_index=True)

fig = px.scatter(df_all_plot, x="Tg (K)", y="Tm (K)", color="group",
                 hover_data=["Name"],
                 title="Separator Safety Prediction: Known vs. Screened Polymers",
                 color_discrete_map={
                     "Predicted safe": "lightgreen",
                     "Predicted unsafe": "lightsalmon",
                     "Known safe": "green",
                     "Known unsafe": "red",
                 })
fig.update_layout(width=700, height=450)
fig.update_xaxes(title_text="Glass Transition Temperature (K)", range=[150, 600], dtick=50)
fig.update_yaxes(title_text="Melting Temperature (K)", range=[280, 650])
fig.show()

---
## Part 4: K-Means Clustering
`clustering_and_evaluation.py` (Part A)

We use unsupervised K-Means clustering to see if polymers naturally group by their material properties, and whether those groups align with safety labels.

**Key concepts:**
- **K-Means** assigns each polymer to the nearest of k cluster centers, then re-computes centers, repeating until stable
- **Elbow plot**: We try k=2 through k=10 and plot inertia (within-cluster sum of squares). The "elbow" where inertia stops dropping steeply suggests a good k
- **Crosstab**: Compares unsupervised clusters to known safety labels — do they align?
- We cluster ALL 140 polymers with complete data (not just the 39 training ones)

In [ ]:
from sklearn.cluster import KMeans

# Use all polymers with complete data
cluster_features = key_features
df_all = df_polymers.dropna(subset=cluster_features).copy()
X_all = df_all[cluster_features]

print(f"Clustering {len(X_all)} polymers with complete data")

### Elbow Plot — choosing k

In [ ]:
# Scale the data
scaler = StandardScaler()
X_all_scaled = scaler.fit_transform(X_all)

# Try different numbers of clusters
ks = range(2, 11)
inertias = []
for k in ks:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_all_scaled)
    inertias.append(model.inertia_)

# Elbow plot
df_elbow = pd.DataFrame({"k": list(ks), "Inertia": inertias})
ax = df_elbow.plot.line(x="k", y="Inertia", marker="o", legend=False)
ax.set_xlabel("k (number of clusters)")
ax.set_ylabel("Inertia (within-cluster sum of squares)")
ax.set_title("Elbow Plot for K-Means Clustering")
plt.show()

### Fit K-Means with k=3 and compare to safety labels

The elbow plot suggests k=3. We use a pipeline (StandardScaler + KMeans) just like we did for KNN, so scaling is consistent.

In [ ]:
# Fit K-Means with k=3 using a pipeline
pipeline_km = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=3, random_state=42, n_init=10)
)
pipeline_km.fit(X_all)

kmeans_model = pipeline_km.named_steps["kmeans"]
df_all["cluster"] = kmeans_model.labels_
df_all["Cluster"] = df_all["cluster"].astype(str)

print("Cluster sizes:")
print(pd.Series(kmeans_model.labels_).value_counts().sort_index())

In [ ]:
# Compare clusters to known safety labels
df_training_clustered = df_training.merge(
    df_all[["Name", "cluster"]],
    left_on="polymer",
    right_on="Name",
    how="left"
)

print("Cluster vs. Safety Label (known separator polymers):")
pd.crosstab(df_training_clustered["cluster"], df_training_clustered["safety"])

### Clusters vs. Safety Labels Plot

All polymers are colored by cluster. Known safe polymers are shown as squares, known unsafe as X markers. This reveals whether unsupervised groupings capture the safe/unsafe distinction.

In [ ]:
cluster_colors = {"0": "#636EFA", "1": "#EF553B", "2": "#00CC96"}

fig = px.scatter(df_all, x="Tg (K)", y="Tm (K)", color="Cluster",
                 hover_data=["Name"],
                 color_discrete_map=cluster_colors,
                 title="K-Means Clusters vs. Known Safety Labels<br>(Clustered on all 6 properties, visualized in Tg vs Tm space)")
fig.update_traces(marker=dict(size=6, opacity=0.8))
fig.update_xaxes(title_text="Glass Transition Temperature (K)")
fig.update_yaxes(title_text="Melting Temperature (K)")

# Overlay known polymers with different shapes
shape_map = {"safe": "square", "unsafe": "x"}
for label in ["safe", "unsafe"]:
    subset = df_training_clustered[df_training_clustered["safety"] == label]
    subset = subset.dropna(subset=["cluster"])
    colors = [cluster_colors[str(int(c))] for c in subset["cluster"]]
    fig.add_scatter(
        x=subset["Tg (K)"], y=subset["Tm (K)"],
        mode="markers",
        name=f"Known {label}",
        marker=dict(size=8, color=colors,
                    symbol=shape_map[label],
                    line=dict(width=1.5, color="black")),
        text=subset["polymer"], hoverinfo="text")

fig.update_xaxes(range=[df_all["Tg (K)"].min() - 20, 600])
fig.update_yaxes(range=[df_all["Tm (K)"].min() - 20, 650])
fig.show()

---
## Part 5: Cross-Validation Confusion Matrix
`clustering_and_evaluation.py` (Part B & C)

The CV confusion matrix is more honest than training — each polymer is predicted by a model that **never saw it** during training.

**Key concepts:**
- `cross_val_predict()` returns out-of-fold predictions for every sample
- `ConfusionMatrixDisplay` gives us a clean heatmap
- For safety-critical applications: **precision on "safe"** matters most (when we predict safe, we need to be right), and **recall on "unsafe"** matters (we need to catch dangerous polymers)

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score)

# Re-run GridSearchCV to get best model
pipeline_knn = make_pipeline(StandardScaler(), KNeighborsClassifier())
grid_cv = GridSearchCV(
    pipeline_knn,
    param_grid={
        "kneighborsclassifier__n_neighbors": range(1, 12),
        "kneighborsclassifier__metric": ["euclidean", "manhattan"],
    },
    scoring="f1_macro",
    cv=5
)
grid_cv.fit(X_train, y_train)
best_pipeline = grid_cv.best_estimator_

# Get out-of-fold predictions
y_cv_pred = cross_val_predict(best_pipeline, X_train, y_train, cv=5)

labels = ["safe", "unsafe"]
cm_cv = confusion_matrix(y_train, y_cv_pred, labels=labels)
print("CV Confusion Matrix (rows=actual, cols=predicted):")
print(cm_cv)

In [ ]:
# Visualize: confusion matrix + metrics table side by side
fig, (ax_cm, ax_metrics) = plt.subplots(1, 2, figsize=(12, 5),
                                        gridspec_kw={"width_ratios": [1, 0.8]})

# Left: confusion matrix heatmap
disp_cv = ConfusionMatrixDisplay(cm_cv, display_labels=labels)
disp_cv.plot(ax=ax_cm, cmap="Blues", values_format="d")
ax_cm.set_title("Cross-Validation Confusion Matrix", fontsize=14)
ax_cm.set_xlabel("Predicted Label", fontsize=13)
ax_cm.set_ylabel("True Label", fontsize=13)

# Right: precision, recall, F1 table
ax_metrics.axis("off")
cv_metrics_data = []
for label in labels:
    p = precision_score(y_train, y_cv_pred, pos_label=label)
    r = recall_score(y_train, y_cv_pred, pos_label=label)
    f = f1_score(y_train, y_cv_pred, pos_label=label)
    cv_metrics_data.append([label, f"{p:.3f}", f"{r:.3f}", f"{f:.3f}"])

table = ax_metrics.table(
    cellText=cv_metrics_data,
    colLabels=["Class", "Precision", "Recall", "F1 Score"],
    loc="center", cellLoc="center"
)
table.auto_set_font_size(False)
table.set_fontsize(15)
table.scale(1, 2.2)

cv_macro_f1 = f1_score(y_train, y_cv_pred, average="macro")
ax_metrics.set_title("Per-Class Metrics (CV)", fontsize=14)
ax_metrics.text(0.5, 0.15, f"Cross-Val Macro F1 = {cv_macro_f1:.3f}",
                transform=ax_metrics.transAxes,
                ha="center", fontsize=16, fontweight="bold")

fig.suptitle("KNN Separator Safety Classifier \u2014 Cross-Validation Performance",
             fontsize=16, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# Key metrics
safe_precision = precision_score(y_train, y_cv_pred, pos_label="safe")
unsafe_recall = recall_score(y_train, y_cv_pred, pos_label="unsafe")
print(f"Safe precision (CV): {safe_precision:.3f}")
print(f"  -> When the model predicts 'safe', it's correct {safe_precision*100:.0f}% of the time")
print(f"\nUnsafe recall (CV): {unsafe_recall:.3f}")
print(f"  -> The model catches {unsafe_recall*100:.0f}% of unsafe polymers")
print(f"\nAs a screening tool, precision matters most \u2014 missed unsafe polymers")
print(f"would be caught during lab testing before going into a battery.")

---
## Part 6: Top Candidates & Radar Chart
`plot_candidates.py`

We visualize the top predicted candidates and compare their material properties against known high-performance separators (PI and PAN).

**Key concepts:**
- Candidates with `predicted_proba_safe == 1.0` are the highest-confidence picks (all k neighbors were "safe")
- The radar chart normalizes all 6 properties to 0-1 so they're comparable on the same axes
- Comparing candidate shapes to PI/PAN shapes reveals which property profiles are similar

In [ ]:
# Load screening results
df_screen = pd.read_csv("data/101screening_polymers.csv")

# Get candidates predicted safe (>50% probability)
df_top = df_screen[df_screen["predicted_proba_safe"] > 0.5].copy()
df_top = df_top.sort_values("predicted_proba_safe", ascending=False)

# Split by confidence level
top_high = df_top[df_top["predicted_proba_safe"] == 1.0].copy()
top_mod = df_top[df_top["predicted_proba_safe"] < 1.0].copy()

print(f"100% confidence candidates: {len(top_high)}")
print(f"50-99% confidence candidates: {len(top_mod)}")
top_high[["Name", "predicted_proba_safe"] + key_features]

### Top Candidates Plot

In [ ]:
top_high_plot = top_high[["Tg (K)", "Tm (K)", "Name"]].copy()
top_high_plot["group"] = "Top candidates (100% confidence)"

top_mod_plot = top_mod[["Tg (K)", "Tm (K)", "Name"]].copy()
top_mod_plot["group"] = "Candidates (50-99% confidence)"

df_plot = pd.concat([top_high_plot, top_mod_plot], ignore_index=True)

fig = px.scatter(df_plot, x="Tg (K)", y="Tm (K)", color="group",
                 hover_data=["Name"],
                 title="Top Predicted Safe Separator Polymer Candidates",
                 color_discrete_map={
                     "Top candidates (100% confidence)": "dodgerblue",
                     "Candidates (50-99% confidence)": "lightskyblue",
                 })

# Label top candidates
label_offsets = {
    "PEKK": (30, -25), "PEN": (25, 15), "PGA": (-60, -25),
    "polyimides": (30, 10), "PA 6": (-55, 20), "PHB": (-50, -20),
    "P(3HB)": (30, -15), "PLLA": (-55, 15), "PBO": (25, 20),
    "nylon 6": (30, -10),
}

for _, row in top_high.iterrows():
    if row["Name"] in label_offsets:
        ax_off, ay_off = label_offsets[row["Name"]]
        fig.add_annotation(
            x=row["Tg (K)"], y=row["Tm (K)"],
            text=row["Name"], showarrow=True,
            arrowhead=0, ax=ax_off, ay=ay_off,
            font=dict(size=10),
            bgcolor="white", bordercolor="gray", borderpad=2,
        )

fig.update_layout(width=700, height=450)
fig.update_traces(marker=dict(size=9))
fig.update_xaxes(title_text="Glass Transition Temperature (K)", range=[150, 600], dtick=50)
fig.update_yaxes(title_text="Melting Temperature (K)", range=[280, 650])
fig.show()

### Radar Chart — Candidates vs. Known Separators (PI and PAN)

We normalize all 6 properties to 0-1 (min-max within this subset) so they're comparable on the same axes, then use `px.line_polar()` to create a radar chart.

- **PI (Known)** excels thermally (highest Tg, high Td) — safe primarily due to extreme heat resistance
- **PAN (Known)** excels mechanically (highest Young's Modulus) — safe primarily due to rigidity
- Candidates closest to PI on the radar (polyimides, PEKK, PEN) are the strongest bets for lab testing

In [ ]:
radar_features = ["Tg (K)", "Tm (K)", "Td (K)",
                  "Tensile Strength (MPa)", "Young's Modulus (MPa)",
                  "Elongation at Break (%)"]
radar_labels = ["Glass Transition Temp", "Melting Temp",
                "Decomposition Temp", "Tensile Strength",
                "Young's Modulus", "Elongation at Break"]

# Prepare top candidate data
df_training = pd.read_csv("data/39known_polymers(training_data).csv")
top_100 = top_high[top_high["predicted_proba_safe"] == 1.0]
top_for_bar = top_100.drop_duplicates(subset="Name").set_index("Name")

candidate_names = ["polyimides", "PEKK", "PGA", "PEN", "PA 6"]

# Build DataFrame — baselines first so they render behind candidates
radar_rows = []

# Add PI and PAN as known baselines
pi = df_training[df_training["polymer"] == "polyimide"].iloc[0]
radar_rows.append({"polymer": "PI (Known)", **{f: pi[f] for f in radar_features}})

pan = df_training[df_training["polymer"] == "PAN"].iloc[0]
radar_rows.append({"polymer": "PAN (Known)", **{f: pan[f] for f in radar_features}})

for name in candidate_names:
    if name in top_for_bar.index:
        row = top_for_bar.loc[name, radar_features]
        radar_rows.append({"polymer": name, **row.to_dict()})

df_radar = pd.DataFrame(radar_rows)

# Normalize each feature to 0-1
for f in radar_features:
    fmin = df_radar[f].min()
    fmax = df_radar[f].max()
    df_radar[f] = (df_radar[f] - fmin) / (fmax - fmin) if fmax > fmin else 0.5

# Reshape to long format for plotly
df_radar_long = df_radar.melt(id_vars="polymer", value_vars=radar_features,
                              var_name="property", value_name="value")

# Replace column names with readable labels
label_map = dict(zip(radar_features, radar_labels))
df_radar_long["property"] = df_radar_long["property"].map(label_map)

fig = px.line_polar(df_radar_long, r="value", theta="property",
                    color="polymer", line_close=True,
                    color_discrete_map={
                        "PI (Known)": "#404040",
                        "PAN (Known)": "#a0a0a0",
                    },
                    title="Top Candidates vs. Current Separators (Normalized)")
fig.update_traces(fill="toself", opacity=0.6, marker=dict(size=8))
fig.update_layout(width=900, height=650,
                  polar=dict(radialaxis=dict(visible=False),
                             angularaxis=dict(tickfont=dict(size=14))),
                  legend=dict(y=1.15, x=1.15, xanchor="left", font=dict(size=13)),
                  title_font_size=16)
fig.show()

### Radar Chart Analysis

- **PI (Known)** excels thermally (highest Tg, high Td) — safe primarily due to extreme heat resistance
- **PAN (Known)** excels mechanically (highest Young's Modulus) — safe primarily due to rigidity that resists dendrite puncture
- **PEKK and PEN** are the most well-rounded candidates — strong across both thermal and mechanical properties
- **PGA** is mechanically extreme (very high tensile strength and modulus) but thermally weaker
- **PA 6** has high elongation but lower Tg — more flexible, different trade-off

Candidates closest to PI on the radar (polyimides, PEKK, PEN) are the strongest bets for lab testing.